# TensileLite Characterization Testing Tutorial

This notebook is a practical companion for reviewing the current branch. It teaches the branch's characterization-testing patterns with small runnable examples, then connects each pattern to the real TensileLite artifacts.

The notebook is intentionally conservative: optional tools such as `mutmut`, `z3-solver`, CrossHair, Hypothesis, ACTS/PICT, Atheris, and Daikon are detected at runtime. Missing optional tools are reported and replaced with small deterministic fallbacks so the tutorial can still run end to end.

## 0. Environment Smoke

First, prove where the notebook is running and which optional tools are available. The planned execution environment is the `tl-char` container with the repository mounted at `/work`.

In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import platform
import subprocess
import sys

def find_repo_root(start=None):
    start = Path(start or os.getcwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'projects/hipblaslt/tensilelite').exists():
            return candidate
    raise RuntimeError(f'Could not find repo root from {start}')

REPO = find_repo_root()
TL = REPO / 'projects/hipblaslt/tensilelite'
CHAR = TL / 'Tensile/Tests/unit/characterization'
WORK = REPO / 'work/tensilelite-characterization'

def tool_available(module_name):
    return importlib.util.find_spec(module_name) is not None

tool_modules = {
    'pytest': 'pytest',
    'coverage.py': 'coverage',
    'syrupy': 'syrupy',
    'mutmut': 'mutmut',
    'z3-solver': 'z3',
    'CrossHair': 'crosshair',
    'Hypothesis': 'hypothesis',
    'PySMT': 'pysmt',
    'Atheris': 'atheris',
}
tool_status = {name: tool_available(module) for name, module in tool_modules.items()}
tool_status['CodeQL CLI'] = bool(__import__('shutil').which('codeql'))
tool_status['java (ACTS/Daikon prerequisite)'] = bool(__import__('shutil').which('java'))

print('python:', sys.version.split()[0])
print('platform:', platform.platform())
print('cwd:', Path.cwd())
print('repo:', REPO)
print('tensilelite:', TL)
print('characterization:', CHAR)
print('LD_LIBRARY_PATH:', os.environ.get('LD_LIBRARY_PATH', '<unset>'))
print('\nOptional tool status:')
for name, ok in tool_status.items():
    print(f'  {name:34} {"available" if ok else "missing - tutorial will use fallback/skip"}')

required_paths = [
    TL / 'Tensile/Tensile.py',
    TL / 'Tensile/Configuration.py',
    TL / 'Tensile/Common/GlobalParameters.py',
    TL / 'Tensile/CustomYamlLoader.py',
    WORK / 'PLAN-MUTATION-PRODUCTION.md',
    WORK / 'PLAN-PARAMETRIC-CHAOS-WORKFLOW.md',
]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError('\n'.join(missing))
print('\nRequired paths: ok')

### Optional: Install Tutorial Tools Into This Kernel

The notebook can run without these tools by using deterministic fallbacks. If you want the full live demos, set `INSTALL_OPTIONAL_TOOLS = False` in the next cell and run it.

The install command uses `sys.executable -m pip`, so it installs into the Python environment backing the active notebook kernel. If your kernel is a virtual environment, this honors that venv. If your kernel is the container Python, it installs into the container Python.

In [ ]:
INSTALL_OPTIONAL_TOOLS = False  # Change to True only if you want this cell to pip install packages.

optional_pip_packages = [
    'pytest',
    'coverage[toml]>=7.0.0',
    'syrupy',
    'mutmut',
    'z3-solver',
    'crosshair-tool',
    'hypothesis',
    'pysmt',
]

print('Active kernel Python:', sys.executable)
print('This is the environment pip will modify if INSTALL_OPTIONAL_TOOLS=True.')

if INSTALL_OPTIONAL_TOOLS:
    cmd = [sys.executable, '-m', 'pip', 'install', *optional_pip_packages]
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)
    tool_status.update({name: tool_available(module) for name, module in tool_modules.items()})
    print('\nUpdated optional tool status:')
    for name, ok in tool_status.items():
        print(f'  {name:34} {"available" if ok else "missing"}')
else:
    print('Skipped install. Set INSTALL_OPTIONAL_TOOLS=True and rerun this cell to install.')

## 1. The Smallest Characterization Test

A characterization test pins current observable behavior. It does not prove that behavior is correct.

Start with plain assertions because they make the idea obvious: call the legacy function, capture what it does today, and fail loudly if that behavior changes. For very small behavior, plain assertions are usually enough.

The problem appears when the behavior is structured, nested, or verbose. Hand-writing a giant expected dictionary inside the test makes the test hard to review, easy to miscopy, and noisy to update. That is the problem `syrupy` solves for this branch: it lets pytest compare a Python value against an external golden snapshot, gives a focused diff when behavior changes, and makes snapshot updates an explicit review event via `--snapshot-update`.


In [ ]:
def legacy_bucket(x):
    # Deliberately odd current behavior: zero is treated as positive.
    return 'positive' if x >= 0 else 'negative'

# For tiny behavior, an explicit expected value is clear and sufficient.
current_behavior = {x: legacy_bucket(x) for x in [-1, 0, 1]}
expected_current_behavior = {-1: 'negative', 0: 'positive', 1: 'positive'}
assert current_behavior == expected_current_behavior
current_behavior


In [ ]:
def preview(path, max_lines=24):
    # Helper used later for short document excerpts, not for making reviewers read test source.
    path = Path(path)
    if not path.exists():
        return f'<missing: {path}>'
    lines = path.read_text(errors='replace').splitlines()
    return '\n'.join(f'{i+1:4}: {line}' for i, line in enumerate(lines[:max_lines]))

def legacy_solution_descriptor(name, params):
    # A tiny stand-in for the kind of structured behavior TensileLite tests pin.
    return {
        'name': name,
        'derived': {
            'workgroup': params.get('workgroup', [16, 16, 1]),
            'vector_width': params.get('vector_width', 1),
            'uses_activation': bool(params.get('activation')),
        },
        'messages': [
            f"solution={name}",
            f"vw={params.get('vector_width', 1)}",
        ],
    }

structured_behavior = legacy_solution_descriptor('toy_gemm', {'vector_width': 4, 'activation': 'relu'})

# This explicit assertion is still possible, but it is already becoming a wall of expected data.
expected_structured_behavior = {
    'name': 'toy_gemm',
    'derived': {'workgroup': [16, 16, 1], 'vector_width': 4, 'uses_activation': True},
    'messages': ['solution=toy_gemm', 'vw=4'],
}
assert structured_behavior == expected_structured_behavior

syrupy_pattern = "assert legacy_solution_descriptor(...) == snapshot"
syrupy_value = {
    'what_syrupy_compares': 'the Python value produced by the test',
    'where_expected_value_lives': 'a committed __snapshots__/*.ambr golden file',
    'how_changes_are_reviewed': 'pytest shows a diff; maintainer accepts with --snapshot-update only when intentional',
    'why_this_branch_uses_it': 'many legacy behaviors are structured enough that inline expected blobs would hide intent',
}

real_test = CHAR / 'BenchmarkStructs/test_benchmark_structs_char.py'
real_snapshot = CHAR / 'BenchmarkStructs/__snapshots__/test_benchmark_structs_char.ambr'
real_summary = {
    'example_module': 'BenchmarkStructs',
    'test_file': str(real_test.relative_to(REPO)),
    'snapshot_file': str(real_snapshot.relative_to(REPO)),
    'test_count': sum(1 for line in real_test.read_text(errors='replace').splitlines() if line.startswith('def test_')),
    'snapshot_size_bytes': real_snapshot.stat().st_size if real_snapshot.exists() else 0,
    'review_context': 'Use the source file for exact mechanics; use this tutorial to understand why the snapshot exists and what review question it answers.',
}

print('syrupy test shape:', syrupy_pattern)
print('What syrupy gives us:')
print(json.dumps(syrupy_value, indent=2))
print('\nReal TensileLite anchor, summarized instead of dumping source:')
print(json.dumps(real_summary, indent=2))


## 2. Pinning Current Bugs Deliberately

Pinned current behavior may include known defects. That is useful when the goal is safe refactoring: a future semantic fix should change the test deliberately.

In [ ]:
def buggy_any(values):
    # Bug: only checks the first two values.
    if len(values) < 2:
        return any(values)
    return values[0] or values[1]

pinned_bug = buggy_any([False, False, True])
assert pinned_bug is False
print('Pinned current buggy behavior:', pinned_bug)

pr_description = WORK / 'PR-DESCRIPTION.md'
text = pr_description.read_text(errors='replace')
start = text.find('## Latent bugs surfaced')
end = text.find('## The one non-additive change')
print('\nReal branch latent-bug section:')
print(text[start:end].strip()[:1800])

## 3. Snapshot Hygiene: Determinism, Normalization, And Side Effects

A useful golden is stable because the test controls input, environment, ordering, and mutable state.

In [ ]:
from datetime import datetime, timezone

def unstable_record(path):
    return {'path': str(Path(path).resolve()), 'timestamp': datetime.now(timezone.utc).isoformat()}

def normalized_record(path):
    return {'path_name': Path(path).name, 'timestamp': '<normalized>'}

print('Unstable shape:', unstable_record('/tmp/example.yaml'))
print('Stable shape:', normalized_record('/tmp/example.yaml'))

def mutates_input(cfg):
    cfg['derived'] = cfg['value'] * 2
    return None

cfg = {'value': 21}
before = dict(cfg)
ret = mutates_input(cfg)
after = dict(cfg)
assert ret is None
assert after == {'value': 21, 'derived': 42}
print('Return-only snapshot would miss:', {'return': ret})
print('Better snapshot includes before/after:', {'before': before, 'after': after})

## 4. Scaling The Pattern Across Modules

The branch repeats a reviewable unit: target, tests, snapshots, coverage delta, and resistance.

In [ ]:
module = CHAR / 'TensileLogic'
for name in ['target.md', 'coverage-before.txt', 'coverage-after.txt', 'resistance.md']:
    path = module / name
    print(f'\n--- {path.relative_to(REPO)} ---')
    print(preview(path, 12))

test_count = len(list(CHAR.rglob('test*_char.py')))
snapshot_count = len(list(CHAR.rglob('*.ambr')))
print('\nCharacterization test files:', test_count)
print('Snapshot files:', snapshot_count)

## 5. Codegen Characterization: Same Idea, Different Harness

Codegen characterization uses deterministic CPU-side emit paths, bounded fixtures, process isolation, and stable output summaries.

In [ ]:
codegen_dir = CHAR / '_codegen'
fixtures = sorted((codegen_dir / 'data/test_data/_designed').rglob('*.yaml'))[:5]
print('Example designed codegen fixtures:')
for p in fixtures:
    print(' ', p.relative_to(TL))

snapshots = sorted((codegen_dir / '__snapshots__').glob('*.ambr'))[:3]
print('\nExample codegen snapshots:')
for p in snapshots:
    print(' ', p.relative_to(TL))
    print(preview(p, 10))

methodology = WORK / 'coverage-methodology.md'
text = methodology.read_text(errors='replace')
start = text.find('### Step 1')
end = text.find('### Step 2')
print('\nCoverage shard command excerpt:')
print(text[start:end].strip()[:1800])

## 6. Coverage Numbers Without Misleading People

Coverage is useful only with the methodology and denominator attached.

In [ ]:
baseline = WORK / 'BASELINE-AND-PROGRESS.md'
text = baseline.read_text(errors='replace')
for needle in ['develop baseline', 'Final whole-project coverage', 'P4 round 7']:
    idx = text.find(needle)
    print(f'\n--- around {needle!r} ---')
    print(text[max(0, idx-400):idx+800])

coverage_facts = {
    'develop_unit_branch_inclusive': '22.47%',
    'final_methodology_A_branch_inclusive': '80.70%',
    'final_methodology_A_line_only': '83.48%',
}
coverage_facts

## 7. Mutation Testing: From Spot Check To Production Workflow

Coverage says code executed. Mutation testing asks whether the tests would fail if behavior changed.

In [ ]:
mutation_plan = WORK / 'PLAN-MUTATION-PRODUCTION.md'
print('Completed mutation plan excerpt:')
print(preview(mutation_plan, 70))

print('\nHistorical P6 report:')
print(preview(WORK / 'coverage/p6/mutation-report.txt', 20))
print('\nHistorical P7 report:')
print(preview(WORK / 'coverage/p7/survivor-kill-report.txt', 20))

print('\nmutmut availability:', tool_status.get('mutmut'))
if not tool_status.get('mutmut'):
    print('mutmut missing: continuing with safe toy fallback; production plan installs/configures mutmut in its prep phase.')

In [ ]:
first_slice = [
    'Tensile/Common/Utilities.py',
    'Tensile/TensileLogic/ValidChipId.py',
    'Tensile/TensileLogic/ValidMatrixInstruction.py',
    'Tensile/TensileLogic/ValidWorkGroup.py',
    'Tensile/TensileLogic/ValidWorkGroupMappingXCC.py',
]
print('First production mutation slice:')
for p in first_slice:
    print(' ', p, 'exists=', (TL / p).exists())

def classify_limit(x):
    return 'large' if x >= 10 else 'small'

def test_clean(fn):
    assert fn(9) == 'small'
    assert fn(10) == 'large'

def mutant_classify_limit(x):
    return 'large' if x > 10 else 'small'

def rc_for(fn):
    try:
        test_clean(fn)
        return 0
    except AssertionError:
        return 1

base_rc = rc_for(classify_limit)
mut_rc = rc_for(mutant_classify_limit)
revert_ok = rc_for(classify_limit) == 0
assert base_rc == 0 and mut_rc != 0 and revert_ok

verify_row = {
    'mutant_id': 'toy_boundary_ge_to_gt',
    'file': '<temporary-copy-or-mutmut-id>',
    'apply_method': 'toy_in_memory',
    'test_node': 'test_clean',
    'expect_clean_rc': 0,
    'expect_mutant_rc': 'nonzero',
    'revert_assert': 'clean function passes again',
    'observed': {'base_rc': base_rc, 'mut_rc': mut_rc, 'revert_ok': revert_ok},
}
verify_row

In [ ]:
survivor_buckets = [
    {'bucket': 'missing-assertion-strength', 'meaning': 'code reached but boundary/output not pinned', 'action': 'add focused test + verify row'},
    {'bucket': 'wrong-granularity', 'meaning': 'broad char test hits indirectly', 'action': 'add smaller direct behavior test + verify row'},
    {'bucket': 'equivalent', 'meaning': 'semantically same for supported inputs', 'action': 'skeptical audit; accept only if airtight'},
    {'bucket': 'intentionally-unhelpful', 'meaning': 'logging/telemetry/format noise', 'action': 'serial pragma proposal with justification'},
    {'bucket': 'design-smell', 'meaning': 'killer test harder than refactor', 'action': 'refactor recommendation'},
]
print(json.dumps(survivor_buckets, indent=2))

mutation_report_shape = {
    'population': {'killed': 4, 'survived': 2, 'skipped': 0, 'suspicious': 0},
    'gate': 'pilot-report-only',
    'survivors_actionable_percent': 100,
    'focused_tests_added': 2,
    'pragmas_added': 0,
}
print('\nmutation-report.json shape:')
print(json.dumps(mutation_report_shape, indent=2))
print('\nsurvivor-ledger.md header: mutant_id | bucket | rationale | action | verify')
print('recommendations.md header: assertion gaps | equivalent patterns | design smells | next slice')

### Actual slice-1 outcome (completed campaign)

The cell above shows the *schema shape* with illustrative placeholder counts. The cell below shows the **actual certified results** of the slice-1 mutation campaign, run via the dynamic workflow `work/tensilelite-characterization/wf/triage-workflow.js` (27 parallel per-function triage agents → a single serial kill-proof through `wf/mutmut-verify.sh` → one bounded repair round → serial pragma apply → synthesis).

**Headline: survivors 131 → 4.** Disposition: 118 killed by new add-only tests (26 files / 74 test functions), 9 removed by 3 `# pragma: no mutate` markers on I/O-noise lines, and 4 left as genuinely *equivalent*. All 4 remaining were independently verified un-killable, so 100% of non-equivalent covered mutants are killed and the covered mutation score rose 77.5% → 99.3%. These numbers were certified by a fresh full `mutmut` re-run plus a manual equivalence audit of each survivor — not by agent self-report.

In [ ]:
# Actual certified outcome of the slice-1 mutation campaign (Phase 2-4).
# Numbers are embedded literals (self-contained): certified by a FRESH full
# `mutmut run` after the new tests + pragmas, plus a MANUAL equivalence audit of
# every remaining survivor -- NOT taken from agent self-report. Full artifacts
# (mutation-report.json, survivor-ledger.md, recommendations.md, 26 test files)
# live on branch users/davidd-amd/tensilelite-mutation (commit 16dad7de847).

slice1_mutmut_rerun = {
    'before': {'total': 665, 'killed': 450, 'survived': 131, 'no_tests': 84},
    'after':  {'total': 654, 'killed': 566, 'survived': 4,   'no_tests': 84},
}
slice1_disposition = {
    'killed_by_new_tests': 118,   # 26 test files / 74 test fns; each proven base_rc=0 mut_rc=1
    'removed_by_pragma': 9,       # 3 `# pragma: no mutate` lines on I/O-noise
    'equivalent_remaining': 4,    # verified genuinely un-killable
    'still_surviving_non_equivalent': 0,
}
assert sum(slice1_disposition[k] for k in
           ('killed_by_new_tests', 'removed_by_pragma', 'equivalent_remaining')) == 131

slice1_scores = {
    'covered': {'before': 0.775, 'after': 0.993},   # killed / (killed + survived)
    'raw':     {'before': 0.677, 'after': 0.865},   # killed / total (incl. no-test gaps)
    'non_equivalent_covered_killed_after': 1.0,
}
remaining_equivalent = {
    '_cu_count_from_path mutmut_9': 'regex cu->CU under re.IGNORECASE (case-insensitive: no behavior change)',
    '_validateWorkGroupMappingXCC mutmut_14': 'missing-key default -1->+1; 1 is positive, a power of two, and divides any CU count -> both paths early-accept',
    'SpinnyThing.increment mutmut_1': 'default value=1->2 on an UNUSED parameter',
    'isRhel8 mutmut_14': 'open(file, "r")->open(file); "r" is the default mode',
}

print('slice-1 mutmut re-run (certified, survivors 131 -> 4):')
print(json.dumps(slice1_mutmut_rerun, indent=2))
print('\nsurvivor disposition:')
print(json.dumps(slice1_disposition, indent=2))
print('\nmutation score:', json.dumps(slice1_scores))
print('\n4 remaining survivors, all verified genuinely equivalent:')
for k, v in remaining_equivalent.items():
    print(f'  - {k}: {v}')
print('\nNote: the 84 "no tests" mutants are coverage gaps (lines no test exercises),')
print('a different problem from assertion strength -> out of scope for this slice.')

### What mutation testing buys us (two perspectives)

**(1) It measures *test quality*, not test quantity.** Line/branch coverage only proves a line *ran*; it cannot tell you whether any test would *notice* if that line's behavior changed. Slice-1 started from ~80% coverage, yet **131 covered mutants survived** — each is a line the suite executed but never actually pinned. Mutation testing localized every weakness to an exact line *and* the input that separates correct from broken (e.g. `ceilDivide`'s `numerator < 0 or denominator < 0` still passed when flipped to `and`; `hash_combine`'s `shift` kwarg was unpinned). Fixing them turned 118 "executed-but-unchecked" lines into behavior-pinning assertions (suite 110 → 184 tests; covered mutation score **77.5% → 99.3%**) and cleanly separated the genuinely-untestable (4 equivalent), the no-contract noise (9 pragma'd), and the refactor candidates (design smells). That is a far more actionable quality signal than a coverage percentage.

**(2) It makes LLM-generated tests *trustworthy* and blocks fraudulent claims.** An LLM can write a test that looks plausible but is always-true, asserts the wrong thing, or simply *claims* "I killed the mutant" without it being so. Mutation testing replaces trust with a mechanical, deterministic gate: a test counts as a kill **only** if it PASSES on clean source, FAILS on the mutated source, and the source reverts clean — `base_rc==0 and mut_rc!=0 and revert=='ok'`, checked by `wf/mutmut-verify.sh`, never by the model's say-so. In this campaign that gate did real work: of 118 LLM-authored tests, **9 failed it on the first pass** (caught, not accepted — repaired and re-proven); the model's **4 equivalence claims were independently audited** rather than trusted; an **inflated synthesis summary** (counts summing to 133 ≠ 131) was caught and corrected; and the whole result was finally certified by an **independent fresh `mutmut` re-run** immune to any agent self-report. LLMs propose; the deterministic harness disposes. The next cell makes that gate executable and shows it rejecting a fabricated kill.

In [ ]:
# Perspective 2, made executable: the kill-certification gate is what turns an
# LLM's "I killed it" CLAIM into a verified FACT. A kill counts ONLY if the test
# passes on clean source, fails on the mutant, and the source reverts clean.

def certify_kill(base_rc, mut_rc, revert):
    """The ONLY definition of a kill the workflow accepts (see wf/mutmut-verify.sh).
    The LLM does not get a vote: this runs against real return codes."""
    return base_rc == 0 and mut_rc != 0 and revert == 'ok'

# Rows in the shape the verifier emits. The middle row is a FRAUDULENT claim:
# an LLM 'kill' whose test actually passes on the mutant too (mut_rc == 0).
claims = [
    {'mutant': 'ceilDivide__mutmut_1',   'base_rc': 0, 'mut_rc': 1, 'revert': 'ok'},   # genuine kill
    {'mutant': 'bogus_always_true_test', 'base_rc': 0, 'mut_rc': 0, 'revert': 'ok'},   # claim != reality
    {'mutant': 'leaky_test',             'base_rc': 0, 'mut_rc': 1, 'revert': 'LEAK'},  # never reverted
]
for c in claims:
    print(f"{c['mutant']:24} certified_kill={certify_kill(c['base_rc'], c['mut_rc'], c['revert'])}")

# Only the genuine row certifies; the fraudulent and leaky rows are rejected.
assert [certify_kill(c['base_rc'], c['mut_rc'], c['revert']) for c in claims] == [True, False, False]

# What that gate (and the surrounding discipline) actually caught in slice-1 --
# none of this was taken on the model's word:
anti_fraud_evidence = {
    'llm_tests_authored': 118,
    'failed_gate_first_pass_then_repaired': 9,           # rejected as not-real-kills, fixed, re-proven
    'llm_equivalence_claims_independently_audited': 4,    # report's rule: never let the model decide equivalence
    'inflated_synthesis_summary_caught_and_corrected': True,  # agent counts summed 133 != 131; corrected to truth
    'final_result_certified_by_independent_mutmut_rerun': True,
}
print('\nanti-fraud evidence (slice-1):')
print(json.dumps(anti_fraud_evidence, indent=2))

## 8. Parameter/Conditional Study: Final Workflow Tooling In Miniature

The next case-selection study avoids Cartesian explosion by extracting branch predicates, mapping them to public inputs, deriving small domains, and validating witnesses.

In [ ]:
workflow_plan = WORK / 'PLAN-PARAMETRIC-CHAOS-WORKFLOW.md'
print('Workflow plan excerpt:')
print(preview(workflow_plan, 90))

run1_files = [
    'Tensile/Tensile.py',
    'Tensile/Configuration.py',
    'Tensile/Common/GlobalParameters.py',
    'Tensile/CustomYamlLoader.py',
]
print('\nRun 1 source paths:')
for p in run1_files:
    print(' ', p, 'exists=', (TL / p).exists())

PUBLIC_SURFACE = WORK / 'parametric-chaos/PublicInputSurface'
if PUBLIC_SURFACE.exists():
    print('\nExisting workflow execution artifacts detected:', PUBLIC_SURFACE.relative_to(REPO))
    for name in ['preflight.json', 'file_inventory.csv', 'branch_census.jsonl', 'constraints_harvested.jsonl']:
        path = PUBLIC_SURFACE / name
        print(' ', name, 'exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else 0)
else:
    print('\nNo existing workflow execution artifacts found; using tutorial fallbacks only.')

In [ ]:
import ast

source_path = TL / 'Tensile/Tensile.py'
source = source_path.read_text(errors='replace')
tree = ast.parse(source)

class BranchVisitor(ast.NodeVisitor):
    def __init__(self):
        self.records = []
        self.function_stack = []
    def visit_FunctionDef(self, node):
        self.function_stack.append(node.name)
        self.generic_visit(node)
        self.function_stack.pop()
    def visit_If(self, node):
        if 520 <= node.lineno <= 532:
            self.records.append({
                'file': 'Tensile/Tensile.py',
                'function': self.function_stack[-1] if self.function_stack else '<module>',
                'line': node.lineno,
                'branch_kind': 'if',
                'predicate_source': ast.get_source_segment(source, node.test),
                'referenced_symbols': sorted({n.id for n in ast.walk(node.test) if isinstance(n, ast.Name)}),
            })
        self.generic_visit(node)

visitor = BranchVisitor()
visitor.visit(tree)
print('Notebook-local ast extraction around Tensile.py:526/529:')
print(json.dumps(visitor.records, indent=2))

branch_census = PUBLIC_SURFACE / 'branch_census.jsonl'
if branch_census.exists():
    print('\nExisting workflow branch_census.jsonl sample:')
    for line in branch_census.read_text(errors='replace').splitlines()[:5]:
        print(line)

In [ ]:
configuration = TL / 'Tensile/Configuration.py'
config_text = configuration.read_text(errors='replace')
for needle in ['class ExpressionEvaluator', 'def addConstraint', 'def checkConstraints']:
    line_no = config_text[:config_text.find(needle)].count('\n') + 1
    print(f'{needle}: line {line_no}')

sample_constraints = ['a > 5', 'altFormat and config_count > 2']
harvested = []
for expr in sample_constraints:
    parsed = ast.parse(expr, mode='exec')
    harvested.append({'expression': expr, 'ast_type': type(parsed.body[0]).__name__, 'dump': ast.dump(parsed, include_attributes=False)})
print('Notebook-local sample constraint harvest:')
print(json.dumps(harvested, indent=2)[:2000])

constraints_file = PUBLIC_SURFACE / 'constraints_harvested.jsonl'
if constraints_file.exists():
    print('\nExisting workflow constraints_harvested.jsonl sample:')
    for line in constraints_file.read_text(errors='replace').splitlines()[:5]:
        print(line)

In [ ]:
branch_record = {
    'branch_id': 'demo:Tensile/Tensile.py:Tensile:526:if',
    'file': 'Tensile/Tensile.py',
    'function': 'Tensile',
    'location': {'line': 526},
    'predicate_source': 'altFormat and len(configPaths) > 2',
    'public_inputs': [
        {'kind': 'cli', 'name': '--alternate-format'},
        {'kind': 'cli', 'name': 'ConfigFile', 'feature': 'count'},
    ],
    'derived_symbols': [
        {'name': 'altFormat', 'derived_from': 'args.AlternateFormat'},
        {'name': 'configPaths', 'derived_from': 'args.ConfigFile'},
    ],
    'domains': {'altFormat': {'type': 'bool'}, 'ConfigFile_count': {'type': 'int', 'values': [0, 1, 2, 3]}},
    'guarded_effect': 'printExit',
}
print(json.dumps(branch_record, indent=2))

In [ ]:
def alt_format_rejected(alt_format: bool, config_count: int) -> bool:
    return bool(alt_format and config_count > 2)

def default_format_rejected(alt_format: bool, config_count: int) -> bool:
    return bool((not alt_format) and config_count != 1)

bounded_domain = [(a, n) for a in [False, True] for n in [0, 1, 2, 3]]
enum_results = [{'altFormat': a, 'ConfigFile_count': n, 'alt_reject': alt_format_rejected(a, n), 'default_reject': default_format_rejected(a, n)} for a, n in bounded_domain]
print('Bounded enumeration fallback:')
print(json.dumps(enum_results, indent=2))

if tool_status.get('z3-solver'):
    from z3 import Bool, Int, Solver, And, Not, sat
    alt = Bool('altFormat')
    count = Int('ConfigFile_count')
    solver = Solver()
    solver.add(alt == True, count >= 0, count > 2)
    print('\nZ3 SAT example:', solver.check(), solver.model() if solver.check() == sat else '')
else:
    print('\nZ3 unavailable: bounded enumeration above is the notebook fallback.')

assert alt_format_rejected(True, 3) is True
assert alt_format_rejected(True, 2) is False

In [ ]:
if tool_status.get('Hypothesis'):
    from hypothesis import given, strategies as st
    @given(st.booleans(), st.integers(min_value=0, max_value=5))
    def property_matches_expression(alt, count):
        assert alt_format_rejected(alt, count) == bool(alt and count > 2)
    property_matches_expression()
    print('Hypothesis property validation passed.')
else:
    for alt, count in bounded_domain:
        assert alt_format_rejected(alt, count) == bool(alt and count > 2)
    print('Hypothesis unavailable: deterministic-grid fallback passed.')

if tool_status.get('CrossHair'):
    print('CrossHair is available. Production workflow should run it only on extracted pure helpers like alt_format_rejected().')
else:
    print('CrossHair unavailable: documented skip. Scope rule still demonstrated: helper only, not full CLI.')

In [ ]:
covering_model = {
    'parameters': {'altFormat': [False, True], 'ConfigFile_count': [0, 1, 2, 3]},
    'constraints': ['ConfigFile_count >= 0'],
    'generator': 'built-in fallback; replace with ACTS/PICT when enabled',
}
covering_cases = [
    {'altFormat': False, 'ConfigFile_count': 1, 'expected': 'default ok'},
    {'altFormat': False, 'ConfigFile_count': 0, 'expected': 'default reject'},
    {'altFormat': True, 'ConfigFile_count': 2, 'expected': 'alternate ok'},
    {'altFormat': True, 'ConfigFile_count': 3, 'expected': 'alternate reject'},
]
print('covering_array/model.json style:')
print(json.dumps(covering_model, indent=2))
print('\ncovering_array/cases.csv style rows:')
for row in covering_cases:
    print(row)

print('\nVerify/reify sketch: pure helper explains the branch; production workflow verifies a witness against the real Tensile.Tensile(userArgs) path with isolated tmp_path and observable printExit behavior.')

### What this session added — Runs 1–3 complete, and two lessons that generalize

The dynamic workflow has now characterized all three rollout surfaces (every run **add-only**; the
gate is the methodology-A whole-project **no-regression guard**, not a coverage target):

| Run | Surface | Branches | Tests / cases | Witnesses (SAT / UNKNOWN) | Gate |
|----|---------|---------|--------------|---------------------------|------|
| 1 | public-input (`Tensile.py`, `Configuration.py`, `GlobalParameters.py`, `CustomYamlLoader.py`) | — | 16 / 148 ✓ | — | 80.74% |
| 2 | deeper public-input — fs/os/env gates (`Toolchain/Validators.py`, `ClientWriter.py`, `BenchmarkProblems.py`, `LibraryIO.py`) | 277 | 15 / 115 ✓ | 13 / 6 | 80.75% |
| 3 | codegen residue (`SolutionStructs/Solution.py`, `KernelWriter.py`, `KernelWriterAssembly.py`) | 6066 | 20 / 211 ✓ | 19 / 1 | 80.76% |

Two lessons from running it that apply well beyond this branch:

**Lesson A — "correct but intractable" is a real defect; bound time and analyze complexity up front.**
The branch-census extractor was *functionally correct* but called `ast.get_source_segment`, which
re-splits the entire source on every call → **O(file²)** per file. It ran in seconds on ~800-line
inputs and took **~5 minutes** on the 18.8k-line `KernelWriterAssembly.py` — enough to trip a
workflow agent's ~180 s no-progress watchdog and make Run 3 un-runnable. The fix (split once, cache,
slice only the spanned lines) is **O(n)** → **1.7 s**, with **byte-identical** output. Nobody set a
time limit, so nothing flagged it until it sat on the critical path. Takeaway, now encoded in the
`orchestration-plan` skill: every generated algorithm should carry an achievable *"must finish in
xyz on the largest real input"* bound backed by a one-line Big-O estimate — and when you optimize a
hot path, **prove the fast path equals the naive one by diffing on real inputs** (the same
characterization discipline this notebook teaches, turned on your own tooling). The next cell makes
the scaling difference runnable.

**Lesson B — the deterministic helpers are the source of truth, not the LLM agents.**
The workflow's Assemble agents rewrote `constraints_harvested.jsonl` and `covering_array/model.json`
in ad-hoc schemas and miscounted the scorecard. So after every run the **driver re-runs the
deterministic helpers (`harvest_constraints.py`, `covering_array.py`) and `finalize.py`**, which
recompute the joined/numeric deliverables from ground-truth fragments. Trust the static tools and
solvers; treat agent-authored numbers as drafts to recompute. (Same *measure, don't inflate* rule as
§6 coverage and §7 mutation.)

**On the optional tools.** CodeQL / ACTS / PICT / Daikon / Atheris stayed *unavailable* across all
three runs **by design** — the stdlib pairwise fallback satisfied `covering_array/`, so no host
install and no human step were needed (`INSTALL_OPTIONAL_TOOLS` stays `False`). Enabling them
belongs in a **prebuilt Docker image** so the pipeline stays hermetic and human-free; that work is
scoped in `work/tensilelite-characterization/PLAN-DEFERRED-TOOLS-DOCKER.md`.

In [ ]:
# Lesson A, made runnable: the O(n^2) -> O(n) trap the branch-census extractor actually hit.
# `ast.get_source_segment` re-splits the WHOLE source on every call. Called once per branch that
# is quadratic in file size -- fine on small files, ~5 min on the 18.8k-line
# KernelWriterAssembly.py (enough to trip a workflow agent's ~180s no-progress watchdog). Split
# once and cache -> linear, with BYTE-IDENTICAL output. The assert below is the same
# characterization discipline this notebook teaches, turned on our own tooling: prove fast == naive.
import ast, time

def _make_source(n_branches):
    body = "\n".join(f"    if x == {i}:\n        y = {i}" for i in range(n_branches))
    return "def f(x):\n" + body + "\n"

_split = getattr(ast, "_splitlines_no_ff", None) or (lambda s: s.splitlines(keepends=True))

def _seg_cached(lines, nd):                 # byte-identical to ast.get_source_segment(padded=False)
    a, b = nd.lineno - 1, nd.end_lineno - 1
    if b == a:
        return lines[a].encode()[nd.col_offset:nd.end_col_offset].decode()
    first = lines[a].encode()[nd.col_offset:].decode()
    last = lines[b].encode()[:nd.end_col_offset].decode()
    return first + "".join(lines[a + 1:b]) + last

print(f"{'branches':>9} | {'naive (resplit/call)':>22} | {'cached (split once)':>20} | speedup")
for n in (200, 800, 3200):
    src = _make_source(n)
    nodes = [nd for nd in ast.walk(ast.parse(src)) if isinstance(nd, ast.If)]
    t0 = time.perf_counter(); naive = [ast.get_source_segment(src, nd) for nd in nodes]
    t_naive = time.perf_counter() - t0
    lines = _split(src)
    t0 = time.perf_counter(); cached = [_seg_cached(lines, nd) for nd in nodes]
    t_cached = time.perf_counter() - t0
    assert naive == cached, "fast path MUST be byte-identical to the naive one"  # the verification strategy
    print(f"{n:>9} | {t_naive*1000:>19.1f} ms | {t_cached*1000:>17.1f} ms | x{t_naive/max(t_cached,1e-9):.0f}")

print("\nNaive ~quadratic (4x branches -> ~16x time); cached ~linear. Output identical, proven by assert.")
print("Lesson: during planning, state an achievable time bound + Big-O over the LARGEST REAL input.")
print("'Fast enough in the test' is not evidence -- measure on the real input, and prove an")
print("optimization output-equivalent (here: diff naive vs cached) before trusting it.")

## 9. Reviewer Checklist

- Is the input deterministic and minimal enough?
- Does the assertion pin behavior, or only execution?
- If a snapshot changed, is the update intentional and explained?
- Are mutable argument side effects captured?
- Is resistance documented where coverage stops?
- For codegen, is the golden stable under the documented process/toolchain constraints?
- For mutation-driven tests, is there proof of `PASS clean / FAIL mutated / revert ok`?
- For parameter studies, are external-state branches classified honestly as `UNKNOWN` when they cannot be proven statically?
- For any generated tool/algorithm, is there a stated completion-time bound + Big-O over the *largest real* input — and if a hot path was optimized, a proof the fast path is output-identical to the naive one (Lesson A)?
- Are numeric/joined deliverables recomputed by the deterministic helpers, not taken from an LLM agent's self-reported counts (Lesson B)?

This branch is a safety net for refactoring legacy Python, not a formal correctness proof.